
# Experiment 17: Target-Condition Recalibration for Reliable Embedding-Level Drift Detection

## 1. Objective

This experiment investigates whether target-condition threshold recalibration can restore false-alarm control when an embedding-level drift detector is transferred across different synthetic embedding distributions.

Experiment 16 evaluated whole-detector transfer using source-condition reference statistics and frozen source-condition thresholds. Experiment 17 extends that investigation by keeping the source-fitted detector fixed while recalibrating its decision threshold using independent, no-drift target-condition calibration windows.

## 2. Research Question

How does the amount of target-condition calibration data affect false-alarm control, drift detection performance, and threshold stability across isotropic, anisotropic, and correlated embedding distributions?

## 3. Experimental Design

- Detectors: Euclidean distance and Ledoit–Wolf-regularized Mahalanobis distance.
- Distribution conditions: isotropic, anisotropic, and correlated.
- Target false-alarm rate: 5%.
- Target-condition calibration sample sizes: 25, 50, 100, 200, and 500 windows.
- Random seeds: 20.
- Evaluation: independent no-drift control windows and drift windows.
- Baseline: the original frozen source-condition threshold.
- Recalibration: empirical threshold estimation from target-condition no-drift calibration windows.

The detector's source reference statistics remain fixed throughout the experiment. Only its decision threshold changes during target-condition recalibration.

## 4. Evaluation Criteria

1. False-alarm rate on independent target-condition control data.
2. Drift detection rate on independent target-condition drift data.
3. Variation in estimated thresholds across random seeds.
4. Differences between source-threshold transfer and target-condition recalibration.

## 5. Limitations

The experiment uses controlled synthetic distributions and does not establish real-world biometric recognition accuracy, fairness, or deployment readiness. Target-condition calibration is assumed to use representative no-drift data, an assumption that may not hold in operational environments.


In [1]:

# Experiment 17 — Step 2: Configuration

import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.covariance import LedoitWolf

CONFIG = {
    "experiment_id": 17,
    "experiment_name": "Target-Condition Recalibration",
    "random_seed": 202617,
    "n_seeds": 20,
    "embedding_dimension": 64,
    "window_size": 20,
    "reference_samples": 1000,
    "source_calibration_windows": 200,
    "target_calibration_window_sizes": [25, 50, 100, 200, 500],
    "control_windows": 500,
    "evaluation_windows_per_condition": 500,
    "target_false_alarm_rate": 0.05,
    "mean_shift_strength": 0.5,
    "correlation_strength": 0.6,
    "detectors": ["euclidean", "mahalanobis"],
    "conditions": ["isotropic", "anisotropic", "correlated"],
}

OUTPUT_DIR = Path("/content/results/experiment_17")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "experiment_17_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Experiment 17 configuration loaded.")
print(json.dumps(CONFIG, indent=2))
print("\nPython:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Output directory:", OUTPUT_DIR)


Experiment 17 configuration loaded.
{
  "experiment_id": 17,
  "experiment_name": "Target-Condition Recalibration",
  "random_seed": 202617,
  "n_seeds": 20,
  "embedding_dimension": 64,
  "window_size": 20,
  "reference_samples": 1000,
  "source_calibration_windows": 200,
  "target_calibration_window_sizes": [
    25,
    50,
    100,
    200,
    500
  ],
  "control_windows": 500,
  "evaluation_windows_per_condition": 500,
  "target_false_alarm_rate": 0.05,
  "mean_shift_strength": 0.5,
  "correlation_strength": 0.6,
  "detectors": [
    "euclidean",
    "mahalanobis"
  ],
  "conditions": [
    "isotropic",
    "anisotropic",
    "correlated"
  ]
}

Python: 3.13.16
NumPy: 2.1.3
Pandas: 2.2.3
Output directory: /content/results/experiment_17


In [2]:

# Experiment 17 — Step 3: Define and Validate Covariance Conditions

D = CONFIG["embedding_dimension"]
rho = CONFIG["correlation_strength"]

covariances = {
    "isotropic": np.eye(D),
    "anisotropic": np.diag(np.geomspace(0.25, 4.0, D)),
    "correlated": rho ** np.abs(
        np.subtract.outer(np.arange(D), np.arange(D))
    ),
}

condition_diagnostics = []

for condition, covariance in covariances.items():
    eigenvalues = np.linalg.eigvalsh(covariance)

    condition_diagnostics.append({
        "condition": condition,
        "shape": covariance.shape,
        "finite": bool(np.isfinite(covariance).all()),
        "symmetric": bool(np.allclose(covariance, covariance.T)),
        "minimum_eigenvalue": float(eigenvalues.min()),
        "positive_definite": bool(eigenvalues.min() > 0),
        "condition_number": float(np.linalg.cond(covariance)),
    })

condition_df = pd.DataFrame(condition_diagnostics)
display(condition_df)

assert all(item["finite"] for item in condition_diagnostics)
assert all(item["symmetric"] for item in condition_diagnostics)
assert all(item["positive_definite"] for item in condition_diagnostics)

print("\nPASS: All three covariance matrices are finite, symmetric, and positive definite.")


,condition,shape,finite,symmetric,minimum_eigenvalue,positive_definite,condition_number
0,isotropic,"(64, 64)",True,True,1.00000,True,1.000000
1,anisotropic,"(64, 64)",True,True,0.25000,True,16.000000
2,correlated,"(64, 64)",True,True,0.25014,True,15.864041



PASS: All three covariance matrices are finite, symmetric, and positive definite.


## Step 4: Generate Reference Datasets

### Objective
Generate independent reference datasets for the three synthetic embedding distributions: isotropic, anisotropic, and correlated.

### Method
- Generate 20 datasets per distribution using reproducible random seeds.
- Each dataset contains 1,000 samples with 64 dimensions.
- Use the covariance matrices validated in Step 3.
- Check dataset dimensions and verify that all generated values are finite.

### Expected Output
A total of 60 reference datasets, with 20 datasets per condition.

### Validation
Confirm that every dataset has shape `(1000, 64)` and contains only finite values before proceeding to the next step.

In [3]:

# Experiment 17 — Step 4: Generate Reference Datasets

reference_datasets = {}

for seed in range(CONFIG["n_seeds"]):
    for condition, covariance in covariances.items():
        rng = np.random.default_rng(
            CONFIG["random_seed"]
            + seed * 100
            + CONFIG["conditions"].index(condition)
        )

        reference = rng.multivariate_normal(
            mean=np.zeros(D),
            cov=covariance,
            size=CONFIG["reference_samples"]
        )

        reference_datasets[(seed, condition)] = reference

reference_summary = pd.DataFrame([
    {
        "seed": seed,
        "condition": condition,
        "shape": data.shape,
        "finite": bool(np.isfinite(data).all()),
    }
    for (seed, condition), data in reference_datasets.items()
])

display(reference_summary.groupby("condition").agg(
    dataset_count=("seed", "count"),
    all_finite=("finite", "all"),
))

assert len(reference_datasets) == 60
assert all(
    data.shape == (1000, 64)
    and np.isfinite(data).all()
    for data in reference_datasets.values()
)

print("PASS: Generated 60 valid reference datasets.")


,dataset_count,all_finite
condition,,
anisotropic,20,True
correlated,20,True
isotropic,20,True


PASS: Generated 60 valid reference datasets.


## Step 5: Fit Source-Condition Drift Detectors

### Objective
Fit the Euclidean and Ledoit–Wolf-regularized Mahalanobis detectors using the reference datasets generated in Step 4.

### Method
- Calculate the reference mean for each dataset.
- Estimate a regularized precision matrix using Ledoit–Wolf covariance estimation.
- Keep the fitted detector parameters fixed during target-condition recalibration.
- Validate parameter dimensions and numerical finiteness.

### Expected Output
Fit two detector configurations for each of the 60 reference datasets, producing 120 configurations.

### Validation
Confirm that all reference means and Mahalanobis precision matrices have the expected dimensions and contain finite values.

In [4]:

# Experiment 17 — Step 5: Fit Source-Condition Detectors

detector_models = {}

for (seed, condition), reference in reference_datasets.items():
    reference_mean = reference.mean(axis=0)
    precision_matrix = LedoitWolf().fit(reference).precision_

    detector_models[(seed, condition, "euclidean")] = {
        "mean": reference_mean,
        "precision": None,
    }

    detector_models[(seed, condition, "mahalanobis")] = {
        "mean": reference_mean,
        "precision": precision_matrix,
    }

    assert reference_mean.shape == (D,)
    assert precision_matrix.shape == (D, D)
    assert np.isfinite(reference_mean).all()
    assert np.isfinite(precision_matrix).all()

expected_models = (
    len(reference_datasets) * len(CONFIG["detectors"])
)

assert len(detector_models) == expected_models

print("Reference datasets:", len(reference_datasets))
print("Fitted detector configurations:", len(detector_models))
print("PASS: All source-condition detector parameters are finite.")


Reference datasets: 60
Fitted detector configurations: 120
PASS: All source-condition detector parameters are finite.


## Step 6: Calibrate Source-Condition Thresholds

### Objective
Establish baseline thresholds using no-drift windows generated under each source condition.

### Method
- Generate 200 source-condition calibration windows for each seed and condition.
- Score the window means using both fitted detectors.
- Set each baseline threshold to the empirical 95th percentile of its calibration scores, corresponding to the nominal 5% false-alarm target.
- Freeze the resulting thresholds for subsequent comparisons.

### Expected Output
A total of 120 source-condition thresholds: 60 reference datasets multiplied by two detectors.

### Validation
Confirm that all calibration scores and thresholds are finite and that each threshold is calculated exclusively from its corresponding source-condition calibration data.

### Important Note
The empirical calibration false-alarm rate is approximate and can differ from 5% on independent control data. Independent controls will be used to measure actual false-alarm performance.

In [5]:

# Experiment 17 — Step 6: Calibrate Source-Condition Thresholds

def score_windows(window_means, model, detector):
    delta = window_means - model["mean"]

    if detector == "euclidean":
        return np.linalg.norm(delta, axis=1)

    if detector == "mahalanobis":
        squared = np.einsum(
            "bi,ij,bj->b",
            delta,
            model["precision"],
            delta,
        )
        return np.sqrt(np.maximum(squared, 0.0))

    raise ValueError(f"Unknown detector: {detector}")


source_thresholds = {}
source_calibration_records = []

for (seed, condition), reference in reference_datasets.items():
    rng = np.random.default_rng(
        CONFIG["random_seed"] + 100_000 + seed * 100
        + CONFIG["conditions"].index(condition)
    )

    calibration_samples = rng.multivariate_normal(
        mean=np.zeros(D),
        cov=covariances[condition],
        size=(
            CONFIG["source_calibration_windows"]
            * CONFIG["window_size"]
        ),
    )

    calibration_windows = calibration_samples.reshape(
        CONFIG["source_calibration_windows"],
        CONFIG["window_size"],
        D,
    ).mean(axis=1)

    for detector in CONFIG["detectors"]:
        key = (seed, condition, detector)
        model = detector_models[key]

        scores = score_windows(
            calibration_windows, model, detector
        )

        threshold = float(np.quantile(
            scores,
            1 - CONFIG["target_false_alarm_rate"],
        ))

        assert np.isfinite(scores).all()
        assert np.isfinite(threshold)

        source_thresholds[key] = threshold

        source_calibration_records.append({
            "seed": seed,
            "condition": condition,
            "detector": detector,
            "n_calibration_windows": len(scores),
            "threshold": threshold,
        })

source_calibration_df = pd.DataFrame(source_calibration_records)
display(source_calibration_df.groupby(
    ["condition", "detector"]
)["threshold"].agg(["mean", "std", "count"]))

assert len(source_thresholds) == 120
assert len(source_calibration_df) == 120

source_calibration_df.to_csv(
    OUTPUT_DIR / "source_calibration_thresholds.csv",
    index=False,
)

print("\nCalibration score rows:", len(source_calibration_records)
      * CONFIG["source_calibration_windows"])
print("Source thresholds:", len(source_thresholds))
print("PASS: All source-condition thresholds are finite.")


mean       std  count
condition   detector                              
anisotropic euclidean    2.494496  0.038170     20
            mahalanobis  2.047456  0.025063     20
correlated  euclidean    2.174279  0.046842     20
            mahalanobis  2.065545  0.021790     20
isotropic   euclidean    2.067201  0.025449     20
            mahalanobis  2.067142  0.027574     20


Calibration score rows: 24000
Source thresholds: 120
PASS: All source-condition thresholds are finite.


## Step 7: Generate Target-Condition Calibration Data

### Objective
Generate independent, no-drift calibration windows under each target distribution to evaluate threshold recalibration with different calibration sample sizes.

### Method
- Use the isotropic, anisotropic, and correlated covariance conditions.
- Generate target-condition calibration windows for sample sizes of 25, 50, 100, 200, and 500.
- Keep the source-fitted detector parameters unchanged.
- Use independent random streams for target calibration data and later evaluation data.
- Use only no-drift target windows for threshold recalibration.

### Expected Output
Target calibration data for every seed, target condition, and requested sample size.

### Validation
Confirm the expected sample counts, window dimensions, and finite values. These calibration windows must not be reused as evaluation data.

In [6]:

# Experiment 17 — Step 7: Generate Target-Condition Calibration Data

target_calibration_data = {}

for seed in range(CONFIG["n_seeds"]):
    for condition, covariance in covariances.items():
        for n_windows in CONFIG["target_calibration_window_sizes"]:
            rng = np.random.default_rng(
                CONFIG["random_seed"]
                + 200_000
                + seed * 10_000
                + CONFIG["conditions"].index(condition) * 1_000
                + n_windows
            )

            samples = rng.multivariate_normal(
                mean=np.zeros(D),
                cov=covariance,
                size=n_windows * CONFIG["window_size"],
            )

            windows = samples.reshape(
                n_windows,
                CONFIG["window_size"],
                D,
            )

            target_calibration_data[
                (seed, condition, n_windows)
            ] = windows.mean(axis=1)

expected_count = (
    CONFIG["n_seeds"]
    * len(CONFIG["conditions"])
    * len(CONFIG["target_calibration_window_sizes"])
)

assert len(target_calibration_data) == expected_count

assert all(
    windows.shape == (n_windows, D)
    and np.isfinite(windows).all()
    for (_, _, n_windows), windows in target_calibration_data.items()
)

calibration_summary = pd.DataFrame([
    {
        "seed": seed,
        "condition": condition,
        "n_windows": n_windows,
        "shape": windows.shape,
        "finite": bool(np.isfinite(windows).all()),
    }
    for (seed, condition, n_windows), windows
    in target_calibration_data.items()
])

display(calibration_summary.groupby(
    ["condition", "n_windows"]
).agg(
    dataset_count=("seed", "count"),
    all_finite=("finite", "all"),
))

print("\nTarget calibration datasets:", len(target_calibration_data))
print("PASS: All target-condition calibration windows are valid.")


dataset_count  all_finite
condition   n_windows                           
anisotropic 25                    20        True
            50                    20        True
            100                   20        True
            200                   20        True
            500                   20        True
correlated  25                    20        True
            50                    20        True
            100                   20        True
            200                   20        True
            500                   20        True
isotropic   25                    20        True
            50                    20        True
            100                   20        True
            200                   20        True
            500                   20        True


Target calibration datasets: 300
PASS: All target-condition calibration windows are valid.


## Step 8: Estimate Target-Condition Recalibrated Thresholds

### Objective
Estimate new decision thresholds from target-condition no-drift calibration windows while keeping each source-fitted detector fixed.

### Method
- Evaluate each source-fitted detector on target-condition calibration windows.
- Calculate the empirical 95th-percentile score for each calibration sample size.
- Compare these recalibrated thresholds against the original source-condition threshold.
- Repeat across all random seeds, source conditions, target conditions, detectors, and calibration sample sizes.

### Expected Output
A threshold table covering all 20 seeds, 3 source conditions, 3 target conditions, 2 detectors, and 5 target calibration sample sizes.

### Validation
Check that the threshold values are finite and that the expected number of threshold records is generated.

### Important Note
The source reference mean and precision matrix remain fixed. Only the threshold is recalibrated. This isolates threshold recalibration from detector refitting.

In [7]:

# Experiment 17 — Step 8: Estimate Target-Condition Thresholds

target_threshold_records = []

for seed in range(CONFIG["n_seeds"]):
    for source_condition in CONFIG["conditions"]:
        for target_condition in CONFIG["conditions"]:
            for detector in CONFIG["detectors"]:
                model_key = (seed, source_condition, detector)
                model = detector_models[model_key]

                for n_windows in CONFIG["target_calibration_window_sizes"]:
                    target_windows = target_calibration_data[
                        (seed, target_condition, n_windows)
                    ]

                    scores = score_windows(
                        target_windows, model, detector
                    )

                    threshold = float(np.quantile(
                        scores,
                        1 - CONFIG["target_false_alarm_rate"],
                    ))

                    source_threshold = source_thresholds[model_key]

                    assert np.isfinite(scores).all()
                    assert np.isfinite(threshold)

                    target_threshold_records.append({
                        "seed": seed,
                        "source_condition": source_condition,
                        "target_condition": target_condition,
                        "detector": detector,
                        "n_calibration_windows": n_windows,
                        "source_threshold": source_threshold,
                        "target_threshold": threshold,
                        "threshold_difference": threshold - source_threshold,
                    })

target_threshold_df = pd.DataFrame(target_threshold_records)

expected_rows = (
    CONFIG["n_seeds"]
    * len(CONFIG["conditions"]) ** 2
    * len(CONFIG["detectors"])
    * len(CONFIG["target_calibration_window_sizes"])
)

assert len(target_threshold_df) == expected_rows
assert np.isfinite(
    target_threshold_df[
        ["source_threshold", "target_threshold", "threshold_difference"]
    ].to_numpy()
).all()

target_threshold_df.to_csv(
    OUTPUT_DIR / "target_recalibrated_thresholds.csv",
    index=False,
)

display(target_threshold_df.groupby(
    ["source_condition", "target_condition", "detector", "n_calibration_windows"]
)["target_threshold"].agg(["mean", "std", "count"]).head(15))

print("\nTarget threshold records:", len(target_threshold_df))
print("Expected records:", expected_rows)
print("PASS: All target-condition thresholds are finite.")


mean  \
source_condition target_condition detector    n_calibration_windows             
anisotropic      anisotropic      euclidean   25                     2.451893   
                                              50                     2.502707   
                                              100                    2.487428   
                                              200                    2.483210   
                                              500                    2.495119   
                                  mahalanobis 25                     2.038826   
                                              50                     2.055640   
                                              100                    2.048276   
                                              200                    2.050555   
                                              500                    2.054688   
                 correlated       euclidean   25                     2.094350   
                                              50                     2.154620   
                                              100                    2.162093   
                                              200                    2.155019   
                                              500                    2.186268   

                                                                          std  \
source_condition target_condition detector    n_calibration_windows             
anisotropic      anisotropic      euclidean   25                     0.070067   
                                              50                     0.071713   
                                              100                    0.059659   
                                              200                    0.035152   
                                              500                    0.028186   
                                  mahalanobis 25                     0.038490   
                                              50                     0.055118   
                                              100                    0.029832   
                                              200                    0.027329   
                                              500                    0.022198   
                 correlated       euclidean   25                     0.092459   
                                              50                     0.065220   
                                              100                    0.045042   
                                              200                    0.030013   
                                              500                    0.019948   

                                                                     count  
source_condition target_condition detector    n_calibration_windows         
anisotropic      anisotropic      euclidean   25                        20  
                                              50                        20  
                                              100                       20  
                                              200                       20  
                                              500                       20  
                                  mahalanobis 25                        20  
                                              50                        20  
                                              100                       20  
                                              200                       20  
                                              500                       20  
                 correlated       euclidean   25                        20  
                                              50                        20  
                                              100                       20  
                                              200                       20  
                                              500                       20


Target threshold records: 1800
Expected records: 1800
PASS: All target-condition thresholds are finite.


## Step 9: Evaluate False-Alarm Rates on Independent Control Data

### Objective
Compare the original source-condition thresholds with target-condition recalibrated thresholds using independent target-condition data containing no drift.

### Method
- Generate 500 independent control windows for every seed and target condition.
- Score each control window using the fixed source-fitted detector.
- Evaluate false-alarm rates under both the original source threshold and each recalibrated target threshold.
- Do not reuse calibration windows for evaluation.

### Evaluation Criteria
- False-alarm rate under the original source threshold.
- False-alarm rate after target-condition recalibration.
- Difference between the two false-alarm rates.
- Variation across random seeds.

### Validation
Confirm that the expected number of control evaluations is produced and that all rates lie between zero and one.

### Important Note
The target-condition recalibration threshold is estimated from no-drift calibration data. This experiment assumes those data represent the target condition adequately; it does not establish that the same assumption will hold in a live biometric system.

In [8]:

# Experiment 17 — Step 9: Independent False-Alarm Evaluation

control_records = []

for seed in range(CONFIG["n_seeds"]):
    for target_condition, covariance in covariances.items():
        rng = np.random.default_rng(
            CONFIG["random_seed"]
            + 300_000
            + seed * 100
            + CONFIG["conditions"].index(target_condition)
        )

        samples = rng.multivariate_normal(
            mean=np.zeros(D),
            cov=covariance,
            size=(
                CONFIG["control_windows"]
                * CONFIG["window_size"]
            ),
        )

        control_windows = samples.reshape(
            CONFIG["control_windows"],
            CONFIG["window_size"],
            D,
        ).mean(axis=1)

        for source_condition in CONFIG["conditions"]:
            for detector in CONFIG["detectors"]:
                model_key = (seed, source_condition, detector)
                model = detector_models[model_key]

                scores = score_windows(
                    control_windows, model, detector
                )

                source_threshold = source_thresholds[model_key]

                for n_windows in CONFIG["target_calibration_window_sizes"]:
                    target_threshold = target_threshold_df.loc[
                        (target_threshold_df["seed"] == seed)
                        & (target_threshold_df["source_condition"] == source_condition)
                        & (target_threshold_df["target_condition"] == target_condition)
                        & (target_threshold_df["detector"] == detector)
                        & (target_threshold_df["n_calibration_windows"] == n_windows),
                        "target_threshold",
                    ].iloc[0]

                    control_records.append({
                        "seed": seed,
                        "source_condition": source_condition,
                        "target_condition": target_condition,
                        "detector": detector,
                        "n_calibration_windows": n_windows,
                        "source_threshold_fpr": float(
                            np.mean(scores > source_threshold)
                        ),
                        "recalibrated_threshold_fpr": float(
                            np.mean(scores > target_threshold)
                        ),
                    })

control_eval_df = pd.DataFrame(control_records)

expected_rows = (
    CONFIG["n_seeds"]
    * len(CONFIG["conditions"]) ** 2
    * len(CONFIG["detectors"])
    * len(CONFIG["target_calibration_window_sizes"])
)

assert len(control_eval_df) == expected_rows
assert control_eval_df[
    ["source_threshold_fpr", "recalibrated_threshold_fpr"]
].apply(lambda col: col.between(0, 1).all()).all()

control_eval_df["fpr_improvement"] = (
    control_eval_df["source_threshold_fpr"]
    - control_eval_df["recalibrated_threshold_fpr"]
)

control_eval_df.to_csv(
    OUTPUT_DIR / "independent_control_fpr_results.csv",
    index=False,
)

display(control_eval_df.groupby(
    ["source_condition", "target_condition", "detector", "n_calibration_windows"]
)[["source_threshold_fpr", "recalibrated_threshold_fpr"]].mean().head(15))

print("\nControl evaluation records:", len(control_eval_df))
print("Expected records:", expected_rows)
print("PASS: Independent control false-alarm rates are within [0, 1].")


source_threshold_fpr  \
source_condition target_condition detector    n_calibration_windows                         
anisotropic      anisotropic      euclidean   25                                   0.0499   
                                              50                                   0.0499   
                                              100                                  0.0499   
                                              200                                  0.0499   
                                              500                                  0.0499   
                                  mahalanobis 25                                   0.0536   
                                              50                                   0.0536   
                                              100                                  0.0536   
                                              200                                  0.0536   
                                              500                                  0.0536   
                 correlated       euclidean   25                                   0.0036   
                                              50                                   0.0036   
                                              100                                  0.0036   
                                              200                                  0.0036   
                                              500                                  0.0036   

                                                                     recalibrated_threshold_fpr  
source_condition target_condition detector    n_calibration_windows                              
anisotropic      anisotropic      euclidean   25                                         0.0750  
                                              50                                         0.0507  
                                              100                                        0.0546  
                                              200                                        0.0556  
                                              500                                        0.0487  
                                  mahalanobis 25                                         0.0620  
                                              50                                         0.0568  
                                              100                                        0.0552  
                                              200                                        0.0496  
                                              500                                        0.0489  
                 correlated       euclidean   25                                         0.1147  
                                              50                                         0.0719  
                                              100                                        0.0659  
                                              200                                        0.0656  
                                              500                                        0.0514


Control evaluation records: 1800
Expected records: 1800
PASS: Independent control false-alarm rates are within [0, 1].


## Step 10: Evaluate Drift Detection on Independent Data

### Objective
Compare drift detection rates using source-condition thresholds and target-condition recalibrated thresholds.

### Method
- Generate independent target-condition evaluation windows containing a controlled mean shift.
- Keep the source-fitted detector parameters fixed.
- Evaluate each window using both the original source threshold and the recalibrated target threshold.
- Use the same evaluation windows for both thresholds to ensure a paired comparison.
- Report detection rates across all source and target conditions, detectors, calibration sample sizes, and random seeds.

### Validation
Confirm that the expected number of evaluation records is generated, all detection rates lie between zero and one, and the evaluation data are independent of calibration data.

### Important Note
A higher detection rate alone does not demonstrate better performance. It must be interpreted alongside the corresponding false-alarm rate.

In [9]:

# Experiment 17 — Step 10: Independent Drift Detection Evaluation

drift_records = []

for seed in range(CONFIG["n_seeds"]):
    for target_condition, covariance in covariances.items():
        rng = np.random.default_rng(
            CONFIG["random_seed"]
            + 400_000
            + seed * 100
            + CONFIG["conditions"].index(target_condition)
        )

        shift = np.zeros(D)
        shift[0] = CONFIG["mean_shift_strength"]

        samples = rng.multivariate_normal(
            mean=shift,
            cov=covariance,
            size=(
                CONFIG["evaluation_windows_per_condition"]
                * CONFIG["window_size"]
            ),
        )

        drift_windows = samples.reshape(
            CONFIG["evaluation_windows_per_condition"],
            CONFIG["window_size"],
            D,
        ).mean(axis=1)

        for source_condition in CONFIG["conditions"]:
            for detector in CONFIG["detectors"]:
                model_key = (seed, source_condition, detector)
                model = detector_models[model_key]

                scores = score_windows(
                    drift_windows, model, detector
                )

                source_threshold = source_thresholds[model_key]

                for n_windows in CONFIG["target_calibration_window_sizes"]:
                    target_threshold = target_threshold_df.loc[
                        (target_threshold_df["seed"] == seed)
                        & (target_threshold_df["source_condition"] == source_condition)
                        & (target_threshold_df["target_condition"] == target_condition)
                        & (target_threshold_df["detector"] == detector)
                        & (target_threshold_df["n_calibration_windows"] == n_windows),
                        "target_threshold",
                    ].iloc[0]

                    drift_records.append({
                        "seed": seed,
                        "source_condition": source_condition,
                        "target_condition": target_condition,
                        "detector": detector,
                        "n_calibration_windows": n_windows,
                        "source_threshold_detection_rate": float(
                            np.mean(scores > source_threshold)
                        ),
                        "recalibrated_threshold_detection_rate": float(
                            np.mean(scores > target_threshold)
                        ),
                    })

drift_eval_df = pd.DataFrame(drift_records)

expected_rows = (
    CONFIG["n_seeds"]
    * len(CONFIG["conditions"]) ** 2
    * len(CONFIG["detectors"])
    * len(CONFIG["target_calibration_window_sizes"])
)

assert len(drift_eval_df) == expected_rows
assert drift_eval_df[
    [
        "source_threshold_detection_rate",
        "recalibrated_threshold_detection_rate",
    ]
].apply(lambda col: col.between(0, 1).all()).all()

drift_eval_df["detection_rate_difference"] = (
    drift_eval_df["recalibrated_threshold_detection_rate"]
    - drift_eval_df["source_threshold_detection_rate"]
)

drift_eval_df.to_csv(
    OUTPUT_DIR / "independent_drift_detection_results.csv",
    index=False,
)

display(drift_eval_df.groupby(
    ["source_condition", "target_condition", "detector", "n_calibration_windows"]
)[
    [
        "source_threshold_detection_rate",
        "recalibrated_threshold_detection_rate",
    ]
].mean().head(15))

print("\nDrift evaluation records:", len(drift_eval_df))
print("Expected records:", expected_rows)
print("PASS: Independent drift detection rates are within [0, 1].")


source_threshold_detection_rate  \
source_condition target_condition detector    n_calibration_windows                                    
anisotropic      anisotropic      euclidean   25                                              0.0797   
                                              50                                              0.0797   
                                              100                                             0.0797   
                                              200                                             0.0797   
                                              500                                             0.0797   
                                  mahalanobis 25                                              0.3174   
                                              50                                              0.3174   
                                              100                                             0.3174   
                                              200                                             0.3174   
                                              500                                             0.3174   
                 correlated       euclidean   25                                              0.0062   
                                              50                                              0.0062   
                                              100                                             0.0062   
                                              200                                             0.0062   
                                              500                                             0.0062   

                                                                     recalibrated_threshold_detection_rate  
source_condition target_condition detector    n_calibration_windows                                         
anisotropic      anisotropic      euclidean   25                                                    0.1158  
                                              50                                                    0.0799  
                                              100                                                   0.0869  
                                              200                                                   0.0860  
                                              500                                                   0.0767  
                                  mahalanobis 25                                                    0.3404  
                                              50                                                    0.3117  
                                              100                                                   0.3180  
                                              200                                                   0.3105  
                                              500                                                   0.3048  
                 correlated       euclidean   25                                                    0.1753  
                                              50                                                    0.1126  
                                              100                                                   0.1022  
                                              200                                                   0.1082  
                                              500                                                   0.0854


Drift evaluation records: 1800
Expected records: 1800
PASS: Independent drift detection rates are within [0, 1].


## Step 11: Joint Comparison of False-Alarms and Drift Detection

### Objective
Assess whether target-condition threshold recalibration improves the trade-off between false-alarm control and drift detection.

### Method
- Join the independent control results and drift-detection results using seed, source condition, target condition, detector, and calibration sample size.
- Compare false-alarm rates and detection rates under source thresholds versus recalibrated thresholds.
- Calculate changes in both metrics.
- Summarize results across seeds for each experimental scenario.

### Evaluation Criteria
- Mean false-alarm rate before and after recalibration.
- Mean detection rate before and after recalibration.
- False-alarm and detection-rate differences.
- Whether any apparent detection improvement comes with an unacceptable increase in false alarms.

### Validation
Ensure each control result matches exactly one corresponding drift result and that all comparison metrics are finite.

In [10]:

# Experiment 17 — Step 11: Joint False-Alarm and Detection Comparison

join_keys = [
    "seed",
    "source_condition",
    "target_condition",
    "detector",
    "n_calibration_windows",
]

joint_df = control_eval_df.merge(
    drift_eval_df,
    on=join_keys,
    how="inner",
    validate="one_to_one",
)

joint_df["source_threshold_fpr"] = joint_df["source_threshold_fpr"]
joint_df["recalibrated_threshold_fpr"] = (
    joint_df["recalibrated_threshold_fpr"]
)

joint_df["source_threshold_detection_rate"] = (
    joint_df["source_threshold_detection_rate"]
)

joint_df["recalibrated_threshold_detection_rate"] = (
    joint_df["recalibrated_threshold_detection_rate"]
)

joint_df["fpr_change"] = (
    joint_df["recalibrated_threshold_fpr"]
    - joint_df["source_threshold_fpr"]
)

joint_df["detection_rate_change"] = (
    joint_df["recalibrated_threshold_detection_rate"]
    - joint_df["source_threshold_detection_rate"]
)

metric_columns = [
    "source_threshold_fpr",
    "recalibrated_threshold_fpr",
    "source_threshold_detection_rate",
    "recalibrated_threshold_detection_rate",
    "fpr_change",
    "detection_rate_change",
]

assert len(joint_df) == len(control_eval_df) == len(drift_eval_df)
assert np.isfinite(joint_df[metric_columns].to_numpy()).all()

joint_df.to_csv(
    OUTPUT_DIR / "joint_recalibration_comparison.csv",
    index=False,
)

summary_df = joint_df.groupby(
    ["source_condition", "target_condition", "detector", "n_calibration_windows"]
)[metric_columns].mean().reset_index()

display(summary_df.head(15))

print("\nJoined comparison records:", len(joint_df))
print("PASS: Control and drift records matched one-to-one.")
print("PASS: All joint comparison metrics are finite.")


,source_condition,target_condition,detector,n_calibration_windows,source_threshold_fpr,recalibrated_threshold_fpr,source_threshold_detection_rate,recalibrated_threshold_detection_rate,fpr_change,detection_rate_change
0,anisotropic,anisotropic,euclidean,25,0.0499,0.0750,0.0797,0.1158,0.0251,0.0361
1,anisotropic,anisotropic,euclidean,50,0.0499,0.0507,0.0797,0.0799,0.0008,0.0002
2,anisotropic,anisotropic,euclidean,100,0.0499,0.0546,0.0797,0.0869,0.0047,0.0072
3,anisotropic,anisotropic,euclidean,200,0.0499,0.0556,0.0797,0.0860,0.0057,0.0063
4,anisotropic,anisotropic,euclidean,500,0.0499,0.0487,0.0797,0.0767,-0.0012,-0.0030
5,anisotropic,anisotropic,mahalanobis,25,0.0536,0.0620,0.3174,0.3404,0.0084,0.0230
6,anisotropic,anisotropic,mahalanobis,50,0.0536,0.0568,0.3174,0.3117,0.0032,-0.0057
7,anisotropic,anisotropic,mahalanobis,100,0.0536,0.0552,0.3174,0.3180,0.0016,0.0006
8,anisotropic,anisotropic,mahalanobis,200,0.0536,0.0496,0.3174,0.3105,-0.0040,-0.0069
9,anisotropic,anisotropic,mahalanobis,500,0.0536,0.0489,0.3174,0.3048,-0.0047,-0.0126



Joined comparison records: 1800
PASS: Control and drift records matched one-to-one.
PASS: All joint comparison metrics are finite.


## Step 12: Aggregate Results Across Experimental Conditions

### Objective
Summarize the effects of target-condition recalibration across all source and target distributions, detectors, and calibration sample sizes.

### Method
- Aggregate results across the 20 random seeds.
- Calculate the mean and standard deviation of false-alarm rates and detection rates.
- Report the changes caused by target-condition recalibration.
- Compare performance across calibration sample sizes.

### Evaluation Criteria
- False-alarm control.
- Drift detection performance.
- Sensitivity to calibration sample size.
- Differences between Euclidean and Mahalanobis detectors.

### Validation
Confirm that the summary includes all experimental scenarios and that all reported metrics are finite.

### Interpretation
A recalibration strategy should not be considered successful merely because it increases detection rates. Its false-alarm behaviour must also be considered.

In [11]:

# Experiment 17 — Step 12: Aggregate Results Across Conditions

group_columns = [
    "source_condition",
    "target_condition",
    "detector",
    "n_calibration_windows",
]

summary_stats = joint_df.groupby(group_columns).agg(
    source_fpr_mean=("source_threshold_fpr", "mean"),
    source_fpr_sd=("source_threshold_fpr", "std"),
    recalibrated_fpr_mean=("recalibrated_threshold_fpr", "mean"),
    recalibrated_fpr_sd=("recalibrated_threshold_fpr", "std"),
    source_detection_mean=("source_threshold_detection_rate", "mean"),
    source_detection_sd=("source_threshold_detection_rate", "std"),
    recalibrated_detection_mean=(
        "recalibrated_threshold_detection_rate", "mean"
    ),
    recalibrated_detection_sd=(
        "recalibrated_threshold_detection_rate", "std"
    ),
    mean_fpr_change=("fpr_change", "mean"),
    mean_detection_change=("detection_rate_change", "mean"),
).reset_index()

expected_scenarios = (
    len(CONFIG["conditions"]) ** 2
    * len(CONFIG["detectors"])
    * len(CONFIG["target_calibration_window_sizes"])
)

assert len(summary_stats) == expected_scenarios
assert np.isfinite(
    summary_stats.select_dtypes(include=[np.number]).to_numpy()
).all()

summary_stats.to_csv(
    OUTPUT_DIR / "experiment_17_aggregated_summary.csv",
    index=False,
)

display(summary_stats.head(20))

print("\nAggregated scenarios:", len(summary_stats))
print("Expected scenarios:", expected_scenarios)
print("PASS: All aggregated summary metrics are finite.")


,source_condition,target_condition,detector,n_calibration_windows,source_fpr_mean,source_fpr_sd,recalibrated_fpr_mean,recalibrated_fpr_sd,source_detection_mean,source_detection_sd,recalibrated_detection_mean,recalibrated_detection_sd,mean_fpr_change,mean_detection_change
0,anisotropic,anisotropic,euclidean,25,0.0499,0.020194,0.0750,0.044888,0.0797,0.021029,0.1158,0.061801,0.0251,0.0361
1,anisotropic,anisotropic,euclidean,50,0.0499,0.020194,0.0507,0.031645,0.0797,0.021029,0.0799,0.043215,0.0008,0.0002
2,anisotropic,anisotropic,euclidean,100,0.0499,0.020194,0.0546,0.024863,0.0797,0.021029,0.0869,0.035330,0.0047,0.0072
3,anisotropic,anisotropic,euclidean,200,0.0499,0.020194,0.0556,0.019201,0.0797,0.021029,0.0860,0.020406,0.0057,0.0063
4,anisotropic,anisotropic,euclidean,500,0.0499,0.020194,0.0487,0.014426,0.0797,0.021029,0.0767,0.014829,-0.0012,-0.0030
5,anisotropic,anisotropic,mahalanobis,25,0.0536,0.018036,0.0620,0.029842,0.3174,0.050697,0.3404,0.070363,0.0084,0.0230
6,anisotropic,anisotropic,mahalanobis,50,0.0536,0.018036,0.0568,0.037927,0.3174,0.050697,0.3117,0.117905,0.0032,-0.0057
7,anisotropic,anisotropic,mahalanobis,100,0.0536,0.018036,0.0552,0.028339,0.3174,0.050697,0.3180,0.072070,0.0016,0.0006
8,anisotropic,anisotropic,mahalanobis,200,0.0536,0.018036,0.0496,0.019386,0.3174,0.050697,0.3105,0.046904,-0.0040,-0.0069
9,anisotropic,anisotropic,mahalanobis,500,0.0536,0.018036,0.0489,0.010944,0.3174,0.050697,0.3048,0.052429,-0.0047,-0.0126



Aggregated scenarios: 90
Expected scenarios: 90
PASS: All aggregated summary metrics are finite.


## Step 13: Preliminary Findings and Interpretation

### 1. Objective
Document the preliminary findings from target-condition threshold recalibration.

### 2. Main Observations
- Target-condition recalibration changes both false-alarm rates and drift detection rates.
- The size and direction of these changes depend on the source condition, target condition, detector, and calibration sample size.
- Small calibration samples can produce unstable thresholds and elevated false-alarm rates.
- In some cross-condition scenarios, recalibration reduces false alarms substantially but also reduces detection rates.
- In other scenarios, detection rates increase alongside false-alarm rates.

### 3. Interpretation
Threshold recalibration is not uniformly beneficial. Its performance must be assessed using independent control data and independent drift data together. Higher detection rates alone are insufficient evidence of improved detector performance.

### 4. Limitations
- All distributions and drift conditions are synthetic.
- The evaluation uses a single mean-shift direction.
- The results do not establish real-world biometric recognition accuracy, fairness, or deployment readiness.
- The experiment assumes that representative no-drift target-condition calibration data are available.
- Statistical uncertainty and comparisons across all scenarios still require careful interpretation.

### 5. Status
These are preliminary findings based on the current experimental outputs. Final conclusions will be written after the remaining checks and the project-wide review.

## Step 14: Final Numerical Integrity Audit

### Objective
Verify the completeness and numerical validity of the Experiment 17 outputs before saving the final results.

### Checks
- Reference dataset count and dimensions.
- Source detector and threshold counts.
- Target calibration dataset and threshold counts.
- Independent control and drift evaluation record counts.
- Aggregated scenario count.
- Finite values and valid rate ranges.
- One-to-one alignment between control and drift evaluation records.

### Expected Result
All programmed integrity checks should pass. Passing these checks confirms data consistency and numerical integrity, but does not by itself establish scientific validity or real-world applicability.

In [12]:

# Experiment 17 — Step 14: Final Numerical Integrity Audit

audit_checks = {
    "60_reference_datasets": len(reference_datasets) == 60,
    "120_detector_configurations": len(detector_models) == 120,
    "120_source_thresholds": len(source_thresholds) == 120,
    "300_target_calibration_datasets": len(target_calibration_data) == 300,
    "1800_target_threshold_records": len(target_threshold_df) == 1800,
    "1800_control_records": len(control_eval_df) == 1800,
    "1800_drift_records": len(drift_eval_df) == 1800,
    "1800_joint_records": len(joint_df) == 1800,
    "90_aggregated_scenarios": len(summary_stats) == 90,
    "finite_reference_data": all(
        np.isfinite(x).all() for x in reference_datasets.values()
    ),
    "finite_thresholds": np.isfinite(
        target_threshold_df["target_threshold"].to_numpy()
    ).all(),
    "valid_control_rates": control_eval_df[
        ["source_threshold_fpr", "recalibrated_threshold_fpr"]
    ].apply(lambda col: col.between(0, 1).all()).all(),
    "valid_detection_rates": drift_eval_df[
        [
            "source_threshold_detection_rate",
            "recalibrated_threshold_detection_rate",
        ]
    ].apply(lambda col: col.between(0, 1).all()).all(),
    "matched_control_drift_rows": len(control_eval_df) == len(drift_eval_df)
        == len(joint_df),
    "finite_aggregated_metrics": np.isfinite(
        summary_stats.select_dtypes(include=[np.number]).to_numpy()
    ).all(),
}

audit_df = pd.DataFrame([
    {"check": name, "passed": bool(passed)}
    for name, passed in audit_checks.items()
])

display(audit_df)

passed_count = int(audit_df["passed"].sum())
total_count = len(audit_df)

audit_df.to_csv(
    OUTPUT_DIR / "experiment_17_integrity_audit.csv",
    index=False,
)

print(f"\nIntegrity checks passed: {passed_count}/{total_count}")

if passed_count == total_count:
    print("PASS: All programmed integrity checks passed.")
else:
    print("WARNING: One or more integrity checks failed. Inspect them before packaging.")

assert passed_count == total_count, "Integrity audit failed."


,check,passed
0,60_reference_datasets,True
1,120_detector_configurations,True
2,120_source_thresholds,True
3,300_target_calibration_datasets,True
4,1800_target_threshold_records,True
5,1800_control_records,True
6,1800_drift_records,True
7,1800_joint_records,True
8,90_aggregated_scenarios,True
9,finite_reference_data,True



Integrity checks passed: 15/15
PASS: All programmed integrity checks passed.


## Step 15: Reproducibility Manifest

### Objective
Record the configuration, software environment, output files, and integrity-audit status for Experiment 17.

### Contents
- Experiment identifier and random seed.
- Number of seeds, embedding dimension, window size, and calibration sample sizes.
- Distribution conditions and detector types.
- Python, NumPy, and Pandas versions.
- Names of the principal result files.
- Integrity-audit status.

### Purpose
The manifest helps another researcher understand the experiment configuration and identify its principal outputs.

### Limitation
Recording configuration and software versions does not by itself guarantee bit-for-bit reproducibility across all environments.

In [13]:

# Experiment 17 — Step 15: Save Reproducibility Manifest

import sklearn
import sys
from datetime import datetime, timezone

manifest = {
    "experiment_id": CONFIG["experiment_id"],
    "experiment_name": CONFIG["experiment_name"],
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "configuration": CONFIG,
    "environment": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "principal_outputs": [
        "experiment_17_config.json",
        "source_calibration_thresholds.csv",
        "target_recalibrated_thresholds.csv",
        "independent_control_fpr_results.csv",
        "independent_drift_detection_results.csv",
        "joint_recalibration_comparison.csv",
        "experiment_17_aggregated_summary.csv",
        "experiment_17_integrity_audit.csv",
    ],
    "integrity_checks_passed": passed_count,
    "integrity_checks_total": total_count,
    "integrity_audit_passed": passed_count == total_count,
    "scientific_validation_status": (
        "Not established by numerical integrity checks alone"
    ),
}

manifest_path = OUTPUT_DIR / "experiment_17_manifest.json"

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("\nManifest saved to:", manifest_path)


{
  "experiment_id": 17,
  "experiment_name": "Target-Condition Recalibration",
  "created_utc": "2026-10-27T15:07:53.030080+00:00",
  "configuration": {
    "experiment_id": 17,
    "experiment_name": "Target-Condition Recalibration",
    "random_seed": 202617,
    "n_seeds": 20,
    "embedding_dimension": 64,
    "window_size": 20,
    "reference_samples": 1000,
    "source_calibration_windows": 200,
    "target_calibration_window_sizes": [
      25,
      50,
      100,
      200,
      500
    ],
    "control_windows": 500,
    "evaluation_windows_per_condition": 500,
    "target_false_alarm_rate": 0.05,
    "mean_shift_strength": 0.5,
    "correlation_strength": 0.6,
    "detectors": [
      "euclidean",
      "mahalanobis"
    ],
    "conditions": [
      "isotropic",
      "anisotropic",
      "correlated"
    ]
  },
  "environment": {
    "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scikit_learn": "1.6.1"
  },

## Step 16: Package Experiment 17 Results

### Objective
Create a single archive containing the configuration, analysis outputs, integrity audit, and reproducibility manifest.

### Method
- Collect all existing files in the Experiment 17 results directory.
- Create a ZIP archive for convenient downloading and repository upload.
- Verify that the archive contains the expected result files.

### Validation
Confirm that the ZIP archive exists, is non-empty, and includes the principal output files.

### Note
This package preserves the current results. It does not imply that the scientific conclusions have been independently validated.

In [14]:

# Experiment 17 — Step 16: Package Results

import zipfile

ZIP_PATH = Path("/content/experiment_17_results.zip")

files_to_package = sorted(
    path for path in OUTPUT_DIR.iterdir()
    if path.is_file() and path != ZIP_PATH
)

assert files_to_package, "No result files found to package."

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in files_to_package:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archived_names = archive.namelist()
    bad_file = archive.testzip()

assert bad_file is None, f"Archive integrity failure: {bad_file}"
assert ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 0

print("Archive:", ZIP_PATH)
print("Archive size (bytes):", ZIP_PATH.stat().st_size)
print("Files included:", len(archived_names))

for name in archived_names:
    print(" -", name)

print("\nPASS: ZIP archive created and verified.")


Archive: /content/experiment_17_results.zip
Archive size (bytes): 114676
Files included: 9
 - experiment_17_aggregated_summary.csv
 - experiment_17_config.json
 - experiment_17_integrity_audit.csv
 - experiment_17_manifest.json
 - independent_control_fpr_results.csv
 - independent_drift_detection_results.csv
 - joint_recalibration_comparison.csv
 - source_calibration_thresholds.csv
 - target_recalibrated_thresholds.csv

PASS: ZIP archive created and verified.


# Experiment 17: Target-Condition Recalibration for Embedding-Level Drift Detection

## Objective

Evaluate whether recalibrating a fixed embedding-level drift detector's decision threshold using target-condition no-drift calibration data can improve false-alarm control when the detector is transferred between different synthetic distributions.

## Experimental Setup

- **Random seeds:** 20
- **Embedding dimension:** 64
- **Window size:** 20
- **Reference samples per dataset:** 1,000
- **Target calibration sample sizes:** 25, 50, 100, 200, and 500 windows
- **Nominal false-alarm target:** 5%
- **Detectors:** Euclidean distance and Ledoit–Wolf-regularized Mahalanobis distance
- **Distribution conditions:** isotropic, anisotropic, and correlated
- **Drift:** controlled mean shift of strength 0.5 in the first dimension

## Method

The detector's source reference mean and precision matrix remain fixed. The baseline uses the threshold calibrated under the source distribution. The recalibrated version estimates a new empirical 95th-percentile threshold from target-condition no-drift calibration windows.

False-alarm rates are measured on independent no-drift control windows. Detection rates are measured on separate drift windows.

## Results Summary

The preliminary results show that threshold recalibration does not uniformly improve performance.

- In some cross-condition scenarios, recalibration substantially reduces false alarms.
- In other scenarios, recalibration increases false alarms, especially with small calibration samples.
- Detection rates may increase or decrease after recalibration.
- Changes in detection rate must be interpreted together with changes in false-alarm rate.
- Calibration sample size influences threshold estimates and observed performance, but the effect varies by scenario.

The complete numerical results are available in the CSV files produced by the experiment.

## Integrity Checks

The programmed integrity audit passed **15 out of 15 checks**. These checks cover expected record counts, numerical finiteness, valid rate ranges, and alignment of control and drift evaluation records.

Passing these checks establishes basic numerical and data-structure consistency. It does not establish scientific validity by itself.

## Limitations

- The experiment uses synthetic Gaussian distributions rather than real biometric embeddings.
- Only one mean-shift direction is evaluated.
- Target-condition recalibration assumes access to representative no-drift calibration data.
- The results do not establish real-world biometric accuracy, fairness, or deployment readiness.
- Broader statistical analysis and project-wide methodological review remain necessary before making publication-level claims.

## Output Files

- `experiment_17_config.json`
- `experiment_17_manifest.json`
- `experiment_17_integrity_audit.csv`
- `source_calibration_thresholds.csv`
- `target_recalibrated_thresholds.csv`
- `independent_control_fpr_results.csv`
- `independent_drift_detection_results.csv`
- `joint_recalibration_comparison.csv`
- `experiment_17_aggregated_summary.csv`

## Reproducibility

The configuration and software versions are recorded in `experiment_17_manifest.json`. The generated result files are included in the Experiment 17 results archive.

## Status

**Workflow completed; scientific validation remains provisional.** Findings should be described as preliminary synthetic evidence until the full methodological and statistical review is complete.

## Step 18: Scientific Interpretation and Limitations Review

### Objective
Review whether the observed changes in false-alarm rates and detection rates support the experiment's research question.

### Questions
1. Does target-condition recalibration bring false-alarm rates closer to the nominal 5% target?
2. How does the calibration sample size affect false-alarm rates and detection rates?
3. Do the effects differ between Euclidean and Mahalanobis detectors?
4. Are apparent detection improvements accompanied by unacceptable increases in false alarms?
5. Which conclusions are supported by the current aggregated results, and which remain uncertain?

### Interpretation Standard
A higher detection rate is not automatically an improvement. Results must be interpreted jointly with false-alarm rates and variability across random seeds.

### Limitations
This experiment uses synthetic distributions, a single mean-shift direction, and assumed access to representative target-condition no-drift calibration data. Numerical integrity checks do not replace statistical validation or independent replication.

## Step 19: Paired Statistical Analysis Across Random Seeds

### Objective
Quantify the uncertainty in changes caused by target-condition threshold recalibration.

### Method
- Pair source-threshold and recalibrated-threshold results using the same seed, source condition, target condition, detector, and calibration sample size.
- Calculate the mean change in false-alarm rate and detection rate across the 20 seeds.
- Calculate 95% paired t-confidence intervals for both changes.
- Report the number of paired seeds and the direction of the observed changes.

### Interpretation
Confidence intervals quantify uncertainty across the simulated seeds under this experimental design. They do not establish real-world generalisability. Because many scenarios are compared, results should be interpreted cautiously; unadjusted confidence intervals are exploratory rather than confirmatory.

In [15]:

# Experiment 17 — Step 19: Paired Confidence Intervals Across Seeds

from scipy.stats import t

def paired_ci(values, confidence=0.95):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)

    if n < 2:
        return {
            "n_seeds": n,
            "mean_change": float(np.mean(values)) if n else np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
        }

    mean_change = values.mean()
    standard_error = values.std(ddof=1) / np.sqrt(n)
    critical_value = t.ppf(
        (1 + confidence) / 2,
        df=n - 1,
    )

    return {
        "n_seeds": n,
        "mean_change": float(mean_change),
        "ci_lower": float(mean_change - critical_value * standard_error),
        "ci_upper": float(mean_change + critical_value * standard_error),
    }


paired_results = []

for keys, group in joint_df.groupby([
    "source_condition",
    "target_condition",
    "detector",
    "n_calibration_windows",
]):
    fpr_stats = paired_ci(group["fpr_change"])
    detection_stats = paired_ci(group["detection_rate_change"])

    paired_results.append({
        "source_condition": keys[0],
        "target_condition": keys[1],
        "detector": keys[2],
        "n_calibration_windows": keys[3],
        "n_seeds": fpr_stats["n_seeds"],
        "mean_fpr_change": fpr_stats["mean_change"],
        "fpr_ci_lower": fpr_stats["ci_lower"],
        "fpr_ci_upper": fpr_stats["ci_upper"],
        "mean_detection_change": detection_stats["mean_change"],
        "detection_ci_lower": detection_stats["ci_lower"],
        "detection_ci_upper": detection_stats["ci_upper"],
    })

paired_ci_df = pd.DataFrame(paired_results)

assert len(paired_ci_df) == 90
assert (paired_ci_df["n_seeds"] == CONFIG["n_seeds"]).all()
assert np.isfinite(
    paired_ci_df.select_dtypes(include=[np.number]).to_numpy()
).all()

paired_ci_df.to_csv(
    OUTPUT_DIR / "paired_recalibration_confidence_intervals.csv",
    index=False,
)

display(paired_ci_df.head(20))

print("\nPaired comparisons:", len(paired_ci_df))
print("PASS: All 90 scenarios have paired confidence intervals.")
print("Note: Confidence intervals are unadjusted for multiple comparisons.")


,source_condition,target_condition,detector,n_calibration_windows,n_seeds,mean_fpr_change,fpr_ci_lower,fpr_ci_upper,mean_detection_change,detection_ci_lower,detection_ci_upper
0,anisotropic,anisotropic,euclidean,25,20,0.0251,0.002602,0.047598,0.0361,0.002683,0.069517
1,anisotropic,anisotropic,euclidean,50,20,0.0008,-0.012663,0.014263,0.0002,-0.019432,0.019832
2,anisotropic,anisotropic,euclidean,100,20,0.0047,-0.008103,0.017503,0.0072,-0.011868,0.026268
3,anisotropic,anisotropic,euclidean,200,20,0.0057,-0.004582,0.015982,0.0063,-0.007825,0.020425
4,anisotropic,anisotropic,euclidean,500,20,-0.0012,-0.009034,0.006634,-0.0030,-0.012068,0.006068
5,anisotropic,anisotropic,mahalanobis,25,20,0.0084,-0.008850,0.025650,0.0230,-0.022112,0.068112
6,anisotropic,anisotropic,mahalanobis,50,20,0.0032,-0.013298,0.019698,-0.0057,-0.064127,0.052727
7,anisotropic,anisotropic,mahalanobis,100,20,0.0016,-0.012118,0.015318,0.0006,-0.038234,0.039434
8,anisotropic,anisotropic,mahalanobis,200,20,-0.0040,-0.015290,0.007290,-0.0069,-0.039592,0.025792
9,anisotropic,anisotropic,mahalanobis,500,20,-0.0047,-0.012632,0.003232,-0.0126,-0.038540,0.013340



Paired comparisons: 90
PASS: All 90 scenarios have paired confidence intervals.
Note: Confidence intervals are unadjusted for multiple comparisons.


## Step 20: Paired Uncertainty Analysis — Interpretation

Paired comparisons across 20 random seeds were used to estimate the mean change in false-alarm rate and drift detection rate after target-condition threshold recalibration. Unadjusted 95% t-confidence intervals were calculated for each of 90 scenarios.

The results demonstrate that recalibration effects depend on the source and target distributions, detector, and calibration sample size. In some scenarios, recalibration reduced false alarms while also reducing detection; in others, it increased both rates. Therefore, recalibration cannot be described as universally improving detector performance.

These confidence intervals quantify between-seed uncertainty under the simulated experimental design. They do not establish real-world performance or account for multiple comparisons. Conclusions remain limited to the synthetic distributions, specified drift magnitude, and calibration assumptions used in this experiment.

In [16]:

# Experiment 17 — Step 20: Save and Verify Paired CI Results

ci_path = OUTPUT_DIR / "paired_recalibration_confidence_intervals.csv"

paired_ci_df.to_csv(ci_path, index=False)

assert ci_path.exists()
assert len(paired_ci_df) == 90
assert (paired_ci_df["n_seeds"] == CONFIG["n_seeds"]).all()
assert paired_ci_df[
    ["fpr_ci_lower", "fpr_ci_upper",
     "detection_ci_lower", "detection_ci_upper"]
].notna().all().all()

print("PASS: Paired confidence-interval results saved.")
print("File:", ci_path)
print("Scenario comparisons:", len(paired_ci_df))
print("Seeds per scenario:", CONFIG["n_seeds"])
print("Confidence intervals: finite and present for all scenarios.")


PASS: Paired confidence-interval results saved.
File: /content/results/experiment_17/paired_recalibration_confidence_intervals.csv
Scenario comparisons: 90
Seeds per scenario: 20
Confidence intervals: finite and present for all scenarios.


## Step 21: Finalise the Experiment 17 Results Package

### Objective
Update the experiment archive to include the paired confidence-interval analysis and the latest reproducibility records.

### Expected outputs
- Updated results ZIP archive
- Paired recalibration confidence intervals
- Updated reproducibility manifest
- Verified archive contents

The numerical integrity checks confirm that the expected data structures and results are present. They do not independently establish scientific validity or real-world generalisability.

In [17]:

# Experiment 17 — Step 21: Refresh Manifest and Results Archive

import json
import zipfile
from datetime import datetime, timezone

OUTPUT_DIR = Path("/content/results/experiment_17")
ZIP_PATH = Path("/content/experiment_17_results.zip")
MANIFEST_PATH = OUTPUT_DIR / "experiment_17_manifest.json"

# Refresh manifest with all current experiment outputs.
manifest = {
    "experiment_id": 17,
    "experiment_name": "Target-Condition Recalibration",
    "updated_utc": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG,
    "outputs": sorted(
        path.name for path in OUTPUT_DIR.iterdir()
        if path.is_file()
        and path.name != "experiment_17_manifest.json"
    ),
    "paired_ci_file": "paired_recalibration_confidence_intervals.csv",
    "paired_ci_scenarios": int(len(paired_ci_df)),
    "seeds_per_scenario": int(CONFIG["n_seeds"]),
    "integrity_status": "Passed structural and numerical checks",
    "scientific_validation_status": (
        "Not established by numerical integrity checks alone"
    ),
}

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))

# Rebuild archive from all files in the output directory.
with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

# Verify archive and required new output.
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    names = archive.namelist()
    assert archive.testzip() is None
    assert "paired_recalibration_confidence_intervals.csv" in names
    assert "experiment_17_manifest.json" in names

print("PASS: Manifest refreshed.")
print("PASS: ZIP archive rebuilt and integrity-checked.")
print("Archive:", ZIP_PATH)
print("Files in archive:", len(names))
print("Contains paired confidence intervals: True")
print("Archive size (bytes):", ZIP_PATH.stat().st_size)


PASS: Manifest refreshed.
PASS: ZIP archive rebuilt and integrity-checked.
Archive: /content/experiment_17_results.zip
Files in archive: 10
Contains paired confidence intervals: True
Archive size (bytes): 119511


## Step 22: Prepare Experiment 17 for Repository Publication

The completed Experiment 17 notebook and its associated results will be published to the Real-Time-Embedding-Level-Drift-Detection repository.

The publication package includes the experiment configuration, source and recalibrated thresholds, independent false-alarm and drift-detection results, paired confidence intervals, aggregated summaries, integrity audit, and reproducibility manifest.

The experiment uses synthetic distributions. Its findings should not be interpreted as evidence of real-world biometric recognition accuracy or deployment readiness.

## Step 23: Verify Repository Access

Before publishing Experiment 17, verify the current Colab working directory, Git repository status, and configured remote URL. Confirm that the remote points to the intended Real-Time-Embedding-Level-Drift-Detection repository before staging or committing files.

In [18]:

import os
import subprocess
from pathlib import Path

print("Current directory:", os.getcwd())

result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    repo_path = Path(result.stdout.strip())
    print("Git repository:", repo_path)

    remote = subprocess.run(
        ["git", "remote", "-v"],
        cwd=repo_path,
        capture_output=True,
        text=True
    )
    print("\nConfigured remotes:")
    print(remote.stdout if remote.stdout else "No remotes configured.")

    status = subprocess.run(
        ["git", "status", "--short"],
        cwd=repo_path,
        capture_output=True,
        text=True
    )
    print("\nWorking-tree status:")
    print(status.stdout if status.stdout else "No uncommitted changes.")
else:
    print("This directory is not inside a Git repository.")
    print("Git error:", result.stderr.strip())


Current directory: /content
This directory is not inside a Git repository.
Git error: fatal: not a git repository (or any of the parent directories): .git


In [19]:

from google.colab import files

files.download("/content/experiment_17_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>